# Analysis of PAC comodulogram metrics 

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
from statsmodels.genmod.generalized_estimating_equations import GEE
from statsmodels.genmod.families import Binomial
import rpy2.robjects as robjects
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr 
from matplotlib.lines import Line2D 
import matplotlib as mpl
import matplotlib.pyplot as plt

from scipy.stats import shapiro

In [ ]:
# ─── Load data ────────────────────────────────────────────────────────────────
basepath = Path('/Users/loukia/Library/CloudStorage/Dropbox-UCL/Loukia Katsouri/DataProtocolsEquipment/Ephys_Analysis/RobinData/Analysis/LFP_analysis_single_trials')
open_field_LFP_csv_path = basepath / 'open_field_LFP.csv'
linear_track_LFP_csv_path = basepath / 'linear_track_LFP.csv'
OUTPUT_DIR = basepath
# os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─── Journal / Illustrator global styles ──────────────────────────────────────
mpl.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "Arial",
    "axes.grid": False,
    "font.size": 6,
    "axes.labelsize": 6,
    "axes.titlesize": 6,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "legend.fontsize": 6,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "ytick.left": True,
    "ytick.direction": "in",
    "savefig.transparent": True,
    "savefig.bbox": "tight",
})

GENO_ORDER  = ["WT", "NLGF"]
GENO_COLORS = {"NLGF": "#E07B54", "WT": "#5B8DB8"}

df_open_field = pd.read_csv(open_field_LFP_csv_path)
df_linear = pd.read_csv(linear_track_LFP_csv_path)

# add a new column to indicate the task
df_open_field["environment"] = "Open Field"
df_linear["environment"] = "Linear Track"

# Extract the trial letter before .set
# Example: ...trialb.set -> b
df_open_field["Trial"] = df_open_field["filename"].str.extract(r'([a-zA-Z])\.set$', expand=False)
df_linear["Trial"] = df_linear["filename"].str.extract(r'([a-zA-Z])\.set$', expand=False)

# merge the two dataframes
df = pd.concat([df_open_field, df_linear], ignore_index=True)

# Trial order
df["Trial"] = df["Trial"].astype(str).str.strip().str.lower()
df["environment"] = df["environment"].astype(str).str.strip().str.lower()

conditions = [
    (df["environment"] == "open field") & (df["Trial"] == "b"),
    (df["environment"] == "open field") & (df["Trial"] == "c"),
    (df["environment"] == "linear track") & (df["Trial"] == "a"),
    (df["environment"] == "linear track") & (df["Trial"] == "b"),
]
choices = ["first", "second", "first", "second"]
df["Trial_order"] = np.select(conditions, choices, default=np.nan)

# One row per mouse/environment/trial, with Slow/Fast values side-by-side
id_cols = [
    "mouse_name",
    "environment",
    "Trial",
    "Trial_order",
    "Genotype",
    "Experimenter",
]

# Normalize gamma band labels for robust pivoting
df["gamma_band_norm"] = df["Gamma band"].astype(str).str.strip().str.lower()

# Sanity check: max one row per id + band
check = df.groupby(id_cols + ["gamma_band_norm"]).size()
if (check > 1).any():
    print("Warning: duplicate rows found for some id + gamma_band combinations")

# Pivot both PAC max and mod index from long to wide
wide = (
    df.pivot_table(
        index=id_cols,
        columns="gamma_band_norm",
        values=["theta_gamma_comod_max", "theta_gamma_mod_index", "n_gamma_power_events"],
        aggfunc="first",
    )
)

# Rename MultiIndex columns to flat names
wide.columns = [
    (
        "comodulogram_PAC_slow_gamma" if (metric == "theta_gamma_comod_max" and band == "slow") else
        "comodulogram_PAC_fast_gamma" if (metric == "theta_gamma_comod_max" and band == "fast") else
        "theta_slow_gamma_mod_index" if (metric == "theta_gamma_mod_index" and band == "slow") else
        "theta_fast_gamma_mod_index" if (metric == "theta_gamma_mod_index" and band == "fast") else
        "n_oscillatory_epochs_slow_gamma" if (metric == "n_gamma_power_events" and band == "slow") else
        "n_oscillatory_epochs_fast_gamma" if (metric == "n_gamma_power_events" and band == "fast") else
        f"{metric}_{band}"
    )
    for metric, band in wide.columns
]
wide = wide.reset_index()

# Keep one copy of non-band columns, excluding long-format band-specific fields
other_cols = [
    c for c in df.columns
    if c not in [
        "Gamma band",
        "gamma_band_norm",
        "theta_gamma_comod_max",
        "theta_gamma_mod_index",
        "n_gamma_power_events",
        "comodulogram_PAC_slow_gamma",
        "comodulogram_PAC_fast_gamma",
        "theta_slow_gamma_mod_index",
        "theta_fast_gamma_mod_index",
        "n_oscillatory_epochs_slow_gamma",
        "n_oscillatory_epochs_fast_gamma",
    ]
]
base = df[other_cols].drop_duplicates(subset=id_cols)

# Final deduplicated dataframe
df = base.merge(
    wide[
        id_cols
        + [
            "comodulogram_PAC_slow_gamma",
            "comodulogram_PAC_fast_gamma",
            "theta_slow_gamma_mod_index",
            "theta_fast_gamma_mod_index",
            "n_oscillatory_epochs_slow_gamma",
            "n_oscillatory_epochs_fast_gamma",
        ]
    ],
    on=id_cols,
    how="left",
)

# Clean names / categorical order
df["Genotype"] = pd.Categorical(df["Genotype"], categories=GENO_ORDER, ordered=True)
df["environment"] = df["environment"].astype(str)
df["mouse_name"] = df["mouse_name"].astype(str)
df["Experimenter"] = df["Experimenter"].astype(str)
df["Trial"] = df["Trial"].astype(str)

# save the merged dataframe to a new CSV file
merged_csv_path = basepath / 'merged_LFP_data.csv'
df.to_csv(merged_csv_path, index=False)
print(f"Merged dataframe saved to {merged_csv_path}")
print(f"Dataframe shape: {df.shape}")
print("-------------------------------------------------------------------------------------")
cols = df.columns.tolist()
for i in range(0, len(cols), 4):
    print(cols[i:i+4])
print("-------------------------------------------------------------------------------------")
print(df.head())

In [ ]:
sub = df[(df["environment"] == "linear track") & (df["Trial_order"] == "second")]
print("rows:", len(sub))
print("animals:", sub["mouse_name"].nunique())
print(sub["mouse_name"].value_counts())

## Run normality test for each metric

In [ ]:
# ─── Metrics to analyse ───────────────────────────────────────────────────────

metrics = {
    "power_spectrum_theta_max_freq": "Theta peak frequency (Hz)",
    "comodulogram_PAC_slow_gamma": "Theta–slow gamma PAC",
    "comodulogram_PAC_fast_gamma": "Theta–fast gamma PAC",
    "n_oscillatory_epochs_slow_gamma": "Slow gamma event count",
    "n_oscillatory_epochs_fast_gamma": "Fast gamma event count",
    "theta_slow_gamma_mod_index": "Theta–Slow gamma mod index",
    "theta_fast_gamma_mod_index": "Theta–Fast gamma mod index",
    "theta_running_slope": "Theta–running slope",
    "theta_running_intercept": "Theta–running intercept",
    "theta_running_rvalue": "Theta–running r",
}

metrics = {k: v for k, v in metrics.items() if k in df.columns}

normality_rows = []

for env in sorted(df["environment"].dropna().unique()):
    for trial_order in ["first", "second"]:
        sub = df.loc[
            (df["environment"] == env) & (df["Trial_order"] == trial_order)
        ].copy()

        print(f"\nEnvironment: {env} | Trial: {trial_order} | n = {len(sub)}")

        for metric, label in metrics.items():
            values = sub[metric].dropna()

            if len(values) < 3:
                stat, p_value = np.nan, np.nan
                decision = "too few values"
            else:
                stat, p_value = shapiro(values)
                decision = "Normal" if p_value > 0.05 else "Not normal"

            normality_rows.append({
                "environment": env,
                "trial_order": trial_order,
                "metric": metric,
                "label": label,
                "n": len(values),
                "statistic": stat,
                "p_value": p_value,
                "decision": decision,
            })

            print(
                f"{metric}: statistic={stat:.4f} p={p_value:.4f} -> {decision}"
                if pd.notna(stat) else
                f"{metric}: too few values"
            )

normality_results = pd.DataFrame(normality_rows)
normality_results.to_csv(OUTPUT_DIR / "normality_by_env_and_trial.csv", index=False)
print("\nSaved:", OUTPUT_DIR / "normality_by_env_and_trial.csv")

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.power import TTestIndPower

def _bootstrap_ci95(values, n_boot=2000, seed=10, stat_func=np.nanmedian):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) < 2:
        return np.nan

    rng = np.random.default_rng(seed)
    boot_stats = []
    for _ in range(n_boot):
        sample = rng.choice(values, size=len(values), replace=True)
        boot_stats.append(stat_func(sample))

    return np.percentile(boot_stats, [2.5, 97.5]).tolist()

def _cohens_d(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    na = len(a)
    nb = len(b)
    if na < 2 or nb < 2:
        return np.nan

    sa = np.var(a, ddof=1)
    sb = np.var(b, ddof=1)
    pooled_sd = np.sqrt(((na - 1) * sa + (nb - 1) * sb) / (na + nb - 2))
    if pooled_sd == 0:
        return np.nan
    return (np.mean(a) - np.mean(b)) / pooled_sd

def run_group_comparison(
    df,
    value_col,
    group_col,
    group_a,
    group_b,
    test_type,
    levene_alpha=0.05,
    bootstrap_n=2000,
    bootstrap_seed=10,
):
    sub = df.loc[df[group_col].isin([group_a, group_b]), [group_col, value_col]].copy()
    sub[value_col] = pd.to_numeric(sub[value_col], errors="coerce")
    sub = sub.dropna(subset=[group_col, value_col])

    a = sub.loc[sub[group_col] == group_a, value_col].to_numpy()
    b = sub.loc[sub[group_col] == group_b, value_col].to_numpy()

    n_a = len(a)
    n_b = len(b)

    if n_a == 0 or n_b == 0:
        return {
            "test_used": np.nan,
            "statistic": np.nan,
            "p_value": np.nan,
            "dof": np.nan,
            "effect_size": np.nan,
            "ci95": np.nan,
            "power": np.nan,
            "bf10": np.nan,
            "levene_p": np.nan,
            "equal_var_assumed": np.nan,
            "n_a": n_a,
            "n_b": n_b,
        }

    levene_stat, levene_p = stats.levene(a, b, center="median")
    equal_var_assumed = bool(levene_p >= levene_alpha)

    if test_type == "parametric":
        t_res = stats.ttest_ind(a, b, equal_var=equal_var_assumed, nan_policy="omit")

        statistic = float(t_res.statistic)
        p_value = float(t_res.pvalue)
        dof = float(n_a + n_b - 2) if equal_var_assumed else np.nan
        effect_size = _cohens_d(a, b)

        mean_diff = float(np.mean(a) - np.mean(b))
        se = np.sqrt(np.var(a, ddof=1) / n_a + np.var(b, ddof=1) / n_b)
        if np.isfinite(se) and se > 0:
            if equal_var_assumed:
                crit = stats.t.ppf(0.975, n_a + n_b - 2)
            else:
                s1 = np.var(a, ddof=1) / n_a
                s2 = np.var(b, ddof=1) / n_b
                df_welch = (s1 + s2) ** 2 / ((s1 ** 2) / (n_a - 1) + (s2 ** 2) / (n_b - 1))
                crit = stats.t.ppf(0.975, df_welch)
            ci95 = [mean_diff - crit * se, mean_diff + crit * se]
        else:
            ci95 = np.nan

        if np.isfinite(effect_size) and abs(effect_size) > 0:
            power = TTestIndPower().power(
                effect_size=abs(effect_size),
                nobs1=n_a,
                ratio=n_b / n_a if n_a > 0 else np.nan,
                alpha=0.05,
                alternative="two-sided",
            )
        else:
            power = np.nan

        bf10 = np.nan
        test_used = "welch_ttest" if not equal_var_assumed else "student_ttest"

    elif test_type == "nonparametric":
        mwu = stats.mannwhitneyu(a, b, alternative="two-sided")
        statistic = float(mwu.statistic)
        p_value = float(mwu.pvalue)
        dof = np.nan
        effect_size = 1 - (2 * statistic) / (n_a * n_b)
        ci95 = _bootstrap_ci95(np.concatenate([a, b]), n_boot=bootstrap_n, seed=bootstrap_seed)
        power = np.nan
        bf10 = np.nan
        test_used = "mannwhitney_u"

    else:
        raise ValueError("test_type must be either 'parametric' or 'nonparametric'")

    return {
        "test_used": test_used,
        "statistic": statistic,
        "p_value": p_value,
        "dof": dof,
        "effect_size": effect_size,
        "ci95": ci95,
        "power": power,
        "bf10": bf10,
        "levene_p": float(levene_p),
        "equal_var_assumed": equal_var_assumed,
        "n_a": n_a,
        "n_b": n_b,
    }

In [ ]:
def run_all_comparisons(
    df,
    normality_df,
    value_cols,
    group_col,
    group_a,
    group_b,
    condition_cols,
    normality_alpha=0.05,
):
    """
    For each unique combination of condition_cols in df, determine parametric vs
    non-parametric test from normality_df, then call run_group_comparison.

    normality_df must have columns: environment, trial_order, variable, p, decision
    (matches normality_results.rename(columns={"metric": "variable", "p_value": "p"}))
    """
    rows = []
    test_choice = {}

    conditions = df[condition_cols].drop_duplicates().reset_index(drop=True)

    for _, cond in conditions.iterrows():
        mask = pd.Series(True, index=df.index)
        for col in condition_cols:
            mask &= df[col] == cond[col]
        sub_df = df[mask].copy()

        if len(sub_df) == 0:
            continue

        env_val = cond.get("environment", None)
        trial_order_vals = sub_df["Trial_order"].dropna().unique()
        trial_order_val = trial_order_vals[0] if len(trial_order_vals) > 0 else None

        for value_col in value_cols:
            # Default to non-parametric; switch to parametric if normality holds
            test_type = "nonparametric"

            if normality_df is not None and trial_order_val is not None and env_val is not None:
                norm_row = normality_df[
                    (normality_df["variable"] == value_col)
                    & (normality_df["environment"] == env_val)
                    & (normality_df["trial_order"] == trial_order_val)
                ]
                if len(norm_row) > 0:
                    p_norm = norm_row["p"].iloc[0]
                    if pd.notna(p_norm) and p_norm > normality_alpha:
                        test_type = "parametric"

            result = run_group_comparison(
                df=sub_df,
                value_col=value_col,
                group_col=group_col,
                group_a=group_a,
                group_b=group_b,
                test_type=test_type,
            )

            row = {
                **{col: cond[col] for col in condition_cols},
                "variable": value_col,
                "test_family": test_type,
                **result,
            }
            rows.append(row)

            key = tuple(cond[col] for col in condition_cols) + (value_col,)
            test_choice[key] = test_type

    return pd.DataFrame(rows), test_choice

In [ ]:
results_df, test_choice = run_all_comparisons(
    df=df,
    normality_df=normality_results.rename(columns={"metric": "variable", "p_value": "p"}),
    value_cols=[

        "theta_running_slope",
        "theta_running_intercept",
        "theta_running_rvalue",
        "power_spectrum_theta_max_freq",
        "comodulogram_PAC_slow_gamma",
        "comodulogram_PAC_fast_gamma",
        "n_oscillatory_epochs_slow_gamma",
        "n_oscillatory_epochs_fast_gamma",
        "theta_slow_gamma_mod_index",
        "theta_fast_gamma_mod_index",
    ],
    group_col="Genotype",
    group_a="WT",
    group_b="NLGF",
    condition_cols=["environment", "Trial"],
)

save_path = OUTPUT_DIR / "statistics.csv"
results_df.to_csv(save_path, index=False)
print(f"\nSaved comparison results to {save_path}")

print("\nComparison results:")
print(results_df.head(50))

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

GENO_ORDER = ["WT", "NLGF"]
GENO_COLORS = {"WT": "#5B8DB8", "NLGF": "#E07B54"}

def p_to_stars(p):
    if pd.isna(p):
        return "ns"
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"

def plot_metric_from_results(df, results_df, metric, save=True):
    cond = (
        df[["environment", "Trial"]]
        .drop_duplicates()
        .sort_values(["environment", "Trial"])
        .to_dict("records")
    )

    if len(cond) == 0:
        return

    fig, axes = plt.subplots(1, len(cond), figsize=(2.1 * len(cond), 2.2), sharey=True)
    if len(cond) == 1:
        axes = [axes]

    rng = np.random.default_rng(10)

    for ax, c in zip(axes, cond):
        env = c["environment"]
        trial = c["Trial"]

        sub = df.loc[
            (df["environment"] == env) &
            (df["Trial"] == trial),
            ["Genotype", "mouse_name", metric]
        ].dropna()

        vals = {}
        for g in GENO_ORDER:
            vals[g] = sub.loc[sub["Genotype"].astype(str) == g, metric].to_numpy()

        for xi, g in enumerate(GENO_ORDER):
            y = vals[g]
            if len(y) == 0:
                continue

            q1 = np.nanpercentile(y, 25)
            med = np.nanmedian(y)
            q3 = np.nanpercentile(y, 75)

            ax.hlines([q1, q3], xi - 0.16, xi + 0.16, colors="black", linewidth=0.8, zorder=7)
            ax.hlines(med, xi - 0.24, xi + 0.24, colors="black", linewidth=1.6, zorder=8)

            jitter = rng.normal(0, 0.045, size=len(y))
            ax.scatter(
                np.full(len(y), xi) + jitter,
                y,
                s=14,
                color=GENO_COLORS.get(g, "#aaaaaa"),
                edgecolor="black",
                linewidth=0.35,
                alpha=0.9,
                zorder=10,
            )

        r = results_df.loc[
            (results_df["variable"] == metric) &
            (results_df["environment"] == env) &
            (results_df["Trial"] == trial)
        ]

        if len(r) > 0:
            p = pd.to_numeric(r["p_value"].iloc[0], errors="coerce")
            test_used = str(r["test_used"].iloc[0])

            if sub[metric].notna().any():
                ymax = float(np.nanmax(sub[metric].values))
                ymin = float(np.nanmin(sub[metric].values))
                yrng = (ymax - ymin) if ymax > ymin else max(abs(ymax) * 0.1, 1.0)
                y = ymax + 0.12 * yrng
                h = 0.04 * yrng

                ax.plot([0, 0, 1, 1], [y, y + h, y + h, y], color="black", linewidth=0.8, clip_on=False)
                ax.text(
                    0.5,
                    y + h,
                    f"{p_to_stars(p)} (p={p:.3g}, {test_used})",
                    ha="center",
                    va="bottom",
                    fontsize=7,
                )

        ax.set_xticks(range(len(GENO_ORDER)))
        ax.set_xticklabels(GENO_ORDER)
        ax.set_title(f"{env} | trial {trial}")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    axes[0].set_ylabel(metric)
    fig.tight_layout()

    if save:
        clean_metric = metric.replace("/", "_")
        fig.savefig(os.path.join(OUTPUT_DIR, f"{clean_metric}_by_env_trial_from_results.pdf"))

    plt.show()
    plt.close(fig)

# plot all variables that were tested above
for metric in [m for m in results_df["variable"].dropna().unique() if m in df.columns]:
    plot_metric_from_results(df, results_df, metric, save=True)

## Repeated measures ANOVA 

In [ ]:
# ── Repeated-measures ANOVA (R via rpy2): Trial_order (within) × Genotype (between) ──
# Runs separately for each environment × gamma band combination
import numpy as np
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

ro.r("suppressPackageStartupMessages(library(afex))")

GAMMA_COLS = {
    'slow': 'comodulogram_PAC_slow_gamma',
    'fast': 'comodulogram_PAC_fast_gamma',
}

def sig_label(p):
    if pd.isna(p): return 'NA'
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'

def run_rm_anova_R(long_df):
    """
    Fit PAC ~ Genotype * Trial_order, mouse_name as within-subject ID.
    long_df must have: mouse_name, Genotype, Trial_order, PAC
    """
    with localconverter(ro.default_converter + pandas2ri.converter):
        r_df = ro.conversion.py2rpy(long_df)
    ro.globalenv['dat'] = r_df

    r_code = """
    suppressWarnings({
        dat$mouse_name  <- factor(dat$mouse_name)
        dat$Genotype    <- factor(dat$Genotype,    levels = c("WT", "NLGF"))
        dat$Trial_order <- factor(dat$Trial_order, levels = c("first", "second"))
        dat <- droplevels(dat)

        fit <- aov_ez(
            id      = "mouse_name",
            dv      = "PAC",
            data    = dat,
            between = "Genotype",
            within  = "Trial_order",
            type    = 3,
            anova_table = list(correction = "GG")
        )

        aov_df <- as.data.frame(fit$anova_table)
        aov_df <- data.frame(term = rownames(aov_df), aov_df, row.names = NULL, check.names = FALSE)
        list(anova_table = aov_df)
    })
    """
    out = ro.r(r_code)
    with localconverter(ro.default_converter + pandas2ri.converter):
        aov_df = ro.conversion.rpy2py(out.rx2('anova_table'))
    return aov_df

# ── Run for each environment × gamma band (4 combinations total) ─────────────
rm_anova_results = {}
rm_rows_summary = []

for env in sorted(df['environment'].dropna().unique()):
    for gamma, pac_col in GAMMA_COLS.items():
        label = f"{env} | {gamma} gamma"
        print(f"\n{'='*70}\n{label}\n{'='*70}")

        sub = (
            df[df['environment'] == env]
            [['mouse_name', 'Genotype', 'Trial_order', pac_col]]
            .rename(columns={pac_col: 'PAC'})
            .dropna(subset=['PAC', 'Trial_order'])
            .copy()
        )

        # Balance check: each mouse must have both first and second
        counts = sub.groupby('mouse_name')['Trial_order'].nunique()
        unbalanced = counts[counts != 2]
        if len(unbalanced) > 0:
            print(f"  Unbalanced — dropping mice missing a trial order: {unbalanced.index.tolist()}")
            sub = sub[~sub['mouse_name'].isin(unbalanced.index)]

        n_mice = sub['mouse_name'].nunique()
        print(f"  n rows = {len(sub)}, n mice (balanced) = {n_mice}, "
              f"genotypes = {sorted(sub['Genotype'].astype(str).unique())}")

        if n_mice < 4:
            print("  Skipped: insufficient balanced data")
            continue

        try:
            aov_df = run_rm_anova_R(sub)
            print("\n  RM ANOVA (afex::aov_ez, Type III, GG correction):")
            print(aov_df.to_string(index=False))

            rm_anova_results[(env, gamma)] = aov_df

            p_col = [c for c in aov_df.columns if 'Pr' in c][0]
            for _, row in aov_df.iterrows():
                rm_rows_summary.append({
                    'environment': env,
                    'gamma_band':  gamma,
                    'term':        row['term'],
                    'F':           row.get('F', np.nan),
                    'p_value':     row.get(p_col, np.nan),
                    'sig':         sig_label(row.get(p_col, np.nan)),
                })

            out_path = OUTPUT_DIR / f"PAC_rm_anova_{env}_{gamma}_gamma.csv"
            aov_df.to_csv(out_path, index=False)
            print(f"  Saved: {out_path}")

        except Exception as e:
            print(f"  FAILED: {e}")

df_rm_summary = pd.DataFrame(rm_rows_summary)
summary_path = OUTPUT_DIR / "PAC_rm_anova_summary_all_conditions.csv"
df_rm_summary.to_csv(summary_path, index=False)
print(f"\n✅ Summary saved to: {summary_path}")
print(df_rm_summary.to_string(index=False))

In [ ]:
# ── Mixed models for all metrics ──────────────────────────────────────────────
# Paired (slow/fast): value ~ Genotype * gamma_band * environment * Trial_order + (1|mouse_name)
# Scalar:             value ~ Genotype * environment * Trial_order + (1|mouse_name)

import pandas as pd
import numpy as np
from pathlib import Path
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

ro.r("suppressPackageStartupMessages(library(lme4))")
ro.r("suppressPackageStartupMessages(library(lmerTest))")
ro.r("suppressPackageStartupMessages(library(emmeans))")

def sig_label(p):
    if pd.isna(p): return 'NA'
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'

# ── Metric definitions ────────────────────────────────────────────────────────
PAIRED_METRICS = {k: v for k, v in {
    'PAC': {
        'slow': 'comodulogram_PAC_slow_gamma',
        'fast': 'comodulogram_PAC_fast_gamma',
        'label': 'Theta–gamma PAC',
    },
    'mod_index': {
        'slow': 'theta_slow_gamma_mod_index',
        'fast': 'theta_fast_gamma_mod_index',
        'label': 'Theta–gamma modulation index',
    },
    'n_oscillatory_epochs': {
        'slow': 'n_oscillatory_epochs_slow_gamma',
        'fast': 'n_oscillatory_epochs_fast_gamma',
        'label': 'N oscillatory epochs',
    },
}.items() if v['slow'] in df.columns and v['fast'] in df.columns}

SCALAR_METRICS = {k: v for k, v in {
    'power_spectrum_theta_max_freq': 'Theta peak frequency (Hz)',
    'theta_running_slope':           'Theta–running slope',
    'theta_running_intercept':       'Theta–running intercept',
    'theta_running_rvalue':          'Theta–running r value',
}.items() if k in df.columns}

all_lmer_results = {}

# ── Helper: run R lmer and extract results ────────────────────────────────────
_PAIRED_RCODE = """
suppressWarnings({
    dat$mouse_name  <- factor(dat$mouse_name)
    dat$Genotype    <- factor(dat$Genotype,    levels = c("WT", "NLGF"))
    dat$gamma_band  <- factor(dat$gamma_band,  levels = c("slow", "fast"))
    dat$environment <- factor(dat$environment)
    dat$Trial_order <- factor(dat$Trial_order, levels = c("first", "second"))
    contrasts(dat$Genotype)    <- contr.sum(nlevels(dat$Genotype))
    contrasts(dat$gamma_band)  <- contr.sum(nlevels(dat$gamma_band))
    contrasts(dat$environment) <- contr.sum(nlevels(dat$environment))
    contrasts(dat$Trial_order) <- contr.sum(nlevels(dat$Trial_order))

    fit <- lmer(value ~ Genotype * gamma_band * environment * Trial_order + (1 | mouse_name),
                data = dat, REML = FALSE)

    aov_table <- as.data.frame(anova(fit, type = 3))
    aov_table <- data.frame(term = rownames(aov_table), aov_table, row.names = NULL, check.names = FALSE)

    emm_geno  <- emmeans(fit, ~ Genotype  | gamma_band * environment * Trial_order)
    emm_trial <- emmeans(fit, ~ Trial_order | Genotype * gamma_band * environment)

    list(anova_table    = aov_table,
         geno_contrasts = as.data.frame(contrast(emm_geno,  method = "pairwise")),
         trial_contrasts = as.data.frame(contrast(emm_trial, method = "pairwise")),
         aic    = AIC(fit),
         re_var = as.numeric(VarCorr(fit)$mouse_name[1,1]))
})
"""

_SCALAR_RCODE = """
suppressWarnings({
    dat$mouse_name  <- factor(dat$mouse_name)
    dat$Genotype    <- factor(dat$Genotype,    levels = c("WT", "NLGF"))
    dat$environment <- factor(dat$environment)
    dat$Trial_order <- factor(dat$Trial_order, levels = c("first", "second"))
    contrasts(dat$Genotype)    <- contr.sum(nlevels(dat$Genotype))
    contrasts(dat$environment) <- contr.sum(nlevels(dat$environment))
    contrasts(dat$Trial_order) <- contr.sum(nlevels(dat$Trial_order))

    fit <- lmer(value ~ Genotype * environment * Trial_order + (1 | mouse_name),
                data = dat, REML = FALSE)

    aov_table <- as.data.frame(anova(fit, type = 3))
    aov_table <- data.frame(term = rownames(aov_table), aov_table, row.names = NULL, check.names = FALSE)

    emm_geno  <- emmeans(fit, ~ Genotype  | environment * Trial_order)
    emm_trial <- emmeans(fit, ~ Trial_order | Genotype * environment)

    list(anova_table    = aov_table,
         geno_contrasts = as.data.frame(contrast(emm_geno,  method = "pairwise")),
         trial_contrasts = as.data.frame(contrast(emm_trial, method = "pairwise")),
         aic    = AIC(fit),
         re_var = as.numeric(VarCorr(fit)$mouse_name[1,1]))
})
"""

def _parse_r_output(out):
    with localconverter(ro.default_converter + pandas2ri.converter):
        aov_df          = ro.conversion.rpy2py(out.rx2('anova_table'))
        geno_contrasts  = ro.conversion.rpy2py(out.rx2('geno_contrasts'))
        trial_contrasts = ro.conversion.rpy2py(out.rx2('trial_contrasts'))
    aic    = float(out.rx2('aic')[0])
    re_var = float(out.rx2('re_var')[0])
    return aov_df, geno_contrasts, trial_contrasts, aic, re_var

def _print_results(metric_name, aov_df, geno_contrasts, aic, re_var):
    print(f"\nAIC: {aic:.2f}  |  RE variance (mouse): {re_var:.6f}")
    print("\nType III ANOVA:")
    print(aov_df.to_string(index=False))
    print("\nGenotype contrasts (NLGF vs WT):")
    print(geno_contrasts.to_string(index=False))

def _save_results(out_dir, name, aov_df, geno_contrasts, trial_contrasts):
    aov_df.to_csv(out_dir / f"lmer_{name}_anova.csv", index=False)
    geno_contrasts.to_csv(out_dir / f"lmer_{name}_geno_contrasts.csv", index=False)
    trial_contrasts.to_csv(out_dir / f"lmer_{name}_trial_contrasts.csv", index=False)

# ── Run paired metrics ────────────────────────────────────────────────────────
out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

for metric_name, meta in PAIRED_METRICS.items():
    print(f"\n{'='*70}\nPAIRED: {metric_name}  ({meta['label']})\n{'='*70}")
    try:
        slow = df[['mouse_name', 'Genotype', 'environment', 'Trial_order', meta['slow']]].rename(columns={meta['slow']: 'value'})
        slow['gamma_band'] = 'slow'
        fast = df[['mouse_name', 'Genotype', 'environment', 'Trial_order', meta['fast']]].rename(columns={meta['fast']: 'value'})
        fast['gamma_band'] = 'fast'
        long_df = pd.concat([slow, fast], ignore_index=True).dropna(subset=['value', 'Trial_order'])
        for col in ['mouse_name', 'Genotype', 'environment', 'gamma_band', 'Trial_order']:
            long_df[col] = long_df[col].astype(str)

        print(f"n rows = {len(long_df)}, n mice = {long_df['mouse_name'].nunique()}")

        with localconverter(ro.default_converter + pandas2ri.converter):
            ro.globalenv['dat'] = ro.conversion.py2rpy(long_df)

        out = ro.r(_PAIRED_RCODE)
        aov_df, geno_contrasts, trial_contrasts, aic, re_var = _parse_r_output(out)
        _print_results(metric_name, aov_df, geno_contrasts, aic, re_var)
        _save_results(out_dir, metric_name, aov_df, geno_contrasts, trial_contrasts)
        all_lmer_results[metric_name] = {'anova': aov_df, 'geno_contrasts': geno_contrasts, 'trial_contrasts': trial_contrasts}

    except Exception as e:
        print(f"  FAILED: {e}")

# ── Run scalar metrics ────────────────────────────────────────────────────────
for metric_name, label in SCALAR_METRICS.items():
    print(f"\n{'='*70}\nSCALAR: {metric_name}  ({label})\n{'='*70}")
    try:
        sub = (
            df[['mouse_name', 'Genotype', 'environment', 'Trial_order', metric_name]]
            .rename(columns={metric_name: 'value'})
            .dropna(subset=['value', 'Trial_order'])
            .copy()
        )
        for col in ['mouse_name', 'Genotype', 'environment', 'Trial_order']:
            sub[col] = sub[col].astype(str)

        print(f"n rows = {len(sub)}, n mice = {sub['mouse_name'].nunique()}")

        with localconverter(ro.default_converter + pandas2ri.converter):
            ro.globalenv['dat'] = ro.conversion.py2rpy(sub)

        out = ro.r(_SCALAR_RCODE)
        aov_df, geno_contrasts, trial_contrasts, aic, re_var = _parse_r_output(out)
        _print_results(metric_name, aov_df, geno_contrasts, aic, re_var)
        _save_results(out_dir, metric_name, aov_df, geno_contrasts, trial_contrasts)
        all_lmer_results[metric_name] = {'anova': aov_df, 'geno_contrasts': geno_contrasts, 'trial_contrasts': trial_contrasts}

    except Exception as e:
        print(f"  FAILED: {e}")

print(f"\n✅ All done. Results saved to: {OUTPUT_DIR}")# ── Mixed models for all metrics ──────────────────────────────────────────────
# Paired (slow/fast): value ~ Genotype * gamma_band * environment * Trial_order + (1|mouse_name)
# Scalar:             value ~ Genotype * environment * Trial_order + (1|mouse_name)

import pandas as pd
import numpy as np
from pathlib import Path
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

ro.r("suppressPackageStartupMessages(library(lme4))")
ro.r("suppressPackageStartupMessages(library(lmerTest))")
ro.r("suppressPackageStartupMessages(library(emmeans))")

def sig_label(p):
    if pd.isna(p): return 'NA'
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'

# ── Metric definitions ────────────────────────────────────────────────────────
PAIRED_METRICS = {k: v for k, v in {
    'PAC': {
        'slow': 'comodulogram_PAC_slow_gamma',
        'fast': 'comodulogram_PAC_fast_gamma',
        'label': 'Theta–gamma PAC',
    },
    'mod_index': {
        'slow': 'theta_slow_gamma_mod_index',
        'fast': 'theta_fast_gamma_mod_index',
        'label': 'Theta–gamma modulation index',
    },
    'n_oscillatory_epochs': {
        'slow': 'n_oscillatory_epochs_slow_gamma',
        'fast': 'n_oscillatory_epochs_fast_gamma',
        'label': 'N oscillatory epochs',
    },
}.items() if v['slow'] in df.columns and v['fast'] in df.columns}

SCALAR_METRICS = {k: v for k, v in {
    'power_spectrum_theta_max_freq': 'Theta peak frequency (Hz)',
    'theta_running_slope':           'Theta–running slope',
    'theta_running_intercept':       'Theta–running intercept',
    'theta_running_rvalue':          'Theta–running r value',
}.items() if k in df.columns}

all_lmer_results = {}

# ── Helper: run R lmer and extract results ────────────────────────────────────
_PAIRED_RCODE = """
suppressWarnings({
    dat$mouse_name  <- factor(dat$mouse_name)
    dat$Genotype    <- factor(dat$Genotype,    levels = c("WT", "NLGF"))
    dat$gamma_band  <- factor(dat$gamma_band,  levels = c("slow", "fast"))
    dat$environment <- factor(dat$environment)
    dat$Trial_order <- factor(dat$Trial_order, levels = c("first", "second"))
    contrasts(dat$Genotype)    <- contr.sum(nlevels(dat$Genotype))
    contrasts(dat$gamma_band)  <- contr.sum(nlevels(dat$gamma_band))
    contrasts(dat$environment) <- contr.sum(nlevels(dat$environment))
    contrasts(dat$Trial_order) <- contr.sum(nlevels(dat$Trial_order))

    fit <- lmer(value ~ Genotype * gamma_band * environment * Trial_order + (1 | mouse_name),
                data = dat, REML = FALSE)

    aov_table <- as.data.frame(anova(fit, type = 3))
    aov_table <- data.frame(term = rownames(aov_table), aov_table, row.names = NULL, check.names = FALSE)

    emm_geno  <- emmeans(fit, ~ Genotype  | gamma_band * environment * Trial_order)
    emm_trial <- emmeans(fit, ~ Trial_order | Genotype * gamma_band * environment)

    list(anova_table    = aov_table,
         geno_contrasts = as.data.frame(contrast(emm_geno,  method = "pairwise")),
         trial_contrasts = as.data.frame(contrast(emm_trial, method = "pairwise")),
         aic    = AIC(fit),
         re_var = as.numeric(VarCorr(fit)$mouse_name[1,1]))
})
"""

_SCALAR_RCODE = """
suppressWarnings({
    dat$mouse_name  <- factor(dat$mouse_name)
    dat$Genotype    <- factor(dat$Genotype,    levels = c("WT", "NLGF"))
    dat$environment <- factor(dat$environment)
    dat$Trial_order <- factor(dat$Trial_order, levels = c("first", "second"))
    contrasts(dat$Genotype)    <- contr.sum(nlevels(dat$Genotype))
    contrasts(dat$environment) <- contr.sum(nlevels(dat$environment))
    contrasts(dat$Trial_order) <- contr.sum(nlevels(dat$Trial_order))

    fit <- lmer(value ~ Genotype * environment * Trial_order + (1 | mouse_name),
                data = dat, REML = FALSE)

    aov_table <- as.data.frame(anova(fit, type = 3))
    aov_table <- data.frame(term = rownames(aov_table), aov_table, row.names = NULL, check.names = FALSE)

    emm_geno  <- emmeans(fit, ~ Genotype  | environment * Trial_order)
    emm_trial <- emmeans(fit, ~ Trial_order | Genotype * environment)

    list(anova_table    = aov_table,
         geno_contrasts = as.data.frame(contrast(emm_geno,  method = "pairwise")),
         trial_contrasts = as.data.frame(contrast(emm_trial, method = "pairwise")),
         aic    = AIC(fit),
         re_var = as.numeric(VarCorr(fit)$mouse_name[1,1]))
})
"""

def _parse_r_output(out):
    with localconverter(ro.default_converter + pandas2ri.converter):
        aov_df          = ro.conversion.rpy2py(out.rx2('anova_table'))
        geno_contrasts  = ro.conversion.rpy2py(out.rx2('geno_contrasts'))
        trial_contrasts = ro.conversion.rpy2py(out.rx2('trial_contrasts'))
    aic    = float(out.rx2('aic')[0])
    re_var = float(out.rx2('re_var')[0])
    return aov_df, geno_contrasts, trial_contrasts, aic, re_var

def _print_results(metric_name, aov_df, geno_contrasts, aic, re_var):
    print(f"\nAIC: {aic:.2f}  |  RE variance (mouse): {re_var:.6f}")
    print("\nType III ANOVA:")
    print(aov_df.to_string(index=False))
    print("\nGenotype contrasts (NLGF vs WT):")
    print(geno_contrasts.to_string(index=False))

def _save_results(out_dir, name, aov_df, geno_contrasts, trial_contrasts):
    aov_df.to_csv(out_dir / f"lmer_{name}_anova.csv", index=False)
    geno_contrasts.to_csv(out_dir / f"lmer_{name}_geno_contrasts.csv", index=False)
    trial_contrasts.to_csv(out_dir / f"lmer_{name}_trial_contrasts.csv", index=False)

# ── Run paired metrics ────────────────────────────────────────────────────────
out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

for metric_name, meta in PAIRED_METRICS.items():
    print(f"\n{'='*70}\nPAIRED: {metric_name}  ({meta['label']})\n{'='*70}")
    try:
        slow = df[['mouse_name', 'Genotype', 'environment', 'Trial_order', meta['slow']]].rename(columns={meta['slow']: 'value'})
        slow['gamma_band'] = 'slow'
        fast = df[['mouse_name', 'Genotype', 'environment', 'Trial_order', meta['fast']]].rename(columns={meta['fast']: 'value'})
        fast['gamma_band'] = 'fast'
        long_df = pd.concat([slow, fast], ignore_index=True).dropna(subset=['value', 'Trial_order'])
        for col in ['mouse_name', 'Genotype', 'environment', 'gamma_band', 'Trial_order']:
            long_df[col] = long_df[col].astype(str)

        print(f"n rows = {len(long_df)}, n mice = {long_df['mouse_name'].nunique()}")

        with localconverter(ro.default_converter + pandas2ri.converter):
            ro.globalenv['dat'] = ro.conversion.py2rpy(long_df)

        out = ro.r(_PAIRED_RCODE)
        aov_df, geno_contrasts, trial_contrasts, aic, re_var = _parse_r_output(out)
        _print_results(metric_name, aov_df, geno_contrasts, aic, re_var)
        _save_results(out_dir, metric_name, aov_df, geno_contrasts, trial_contrasts)
        all_lmer_results[metric_name] = {'anova': aov_df, 'geno_contrasts': geno_contrasts, 'trial_contrasts': trial_contrasts}

    except Exception as e:
        print(f"  FAILED: {e}")

# ── Run scalar metrics ────────────────────────────────────────────────────────
for metric_name, label in SCALAR_METRICS.items():
    print(f"\n{'='*70}\nSCALAR: {metric_name}  ({label})\n{'='*70}")
    try:
        sub = (
            df[['mouse_name', 'Genotype', 'environment', 'Trial_order', metric_name]]
            .rename(columns={metric_name: 'value'})
            .dropna(subset=['value', 'Trial_order'])
            .copy()
        )
        for col in ['mouse_name', 'Genotype', 'environment', 'Trial_order']:
            sub[col] = sub[col].astype(str)

        print(f"n rows = {len(sub)}, n mice = {sub['mouse_name'].nunique()}")

        with localconverter(ro.default_converter + pandas2ri.converter):
            ro.globalenv['dat'] = ro.conversion.py2rpy(sub)

        out = ro.r(_SCALAR_RCODE)
        aov_df, geno_contrasts, trial_contrasts, aic, re_var = _parse_r_output(out)
        _print_results(metric_name, aov_df, geno_contrasts, aic, re_var)
        _save_results(out_dir, metric_name, aov_df, geno_contrasts, trial_contrasts)
        all_lmer_results[metric_name] = {'anova': aov_df, 'geno_contrasts': geno_contrasts, 'trial_contrasts': trial_contrasts}

    except Exception as e:
        print(f"  FAILED: {e}")

print(f"\n✅ All done. Results saved to: {OUTPUT_DIR}")

## MLM PAC ~ Genotype * Gamma band * environment

In [ ]:
# ── Full mixed model: PAC ~ Genotype * gamma_band * environment * Trial_order ──
import pandas as pd
import numpy as np
from pathlib import Path
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

ro.r("suppressPackageStartupMessages(library(lme4))")
ro.r("suppressPackageStartupMessages(library(lmerTest))")
ro.r("suppressPackageStartupMessages(library(emmeans))")

SLOW_COL = 'comodulogram_PAC_slow_gamma'
FAST_COL = 'comodulogram_PAC_fast_gamma'

def sig_label(p):
    if pd.isna(p): return 'NA'
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'

# ── Build long-format PAC (both gamma bands, environments, trial orders) ──────
slow = df[['mouse_name', 'Genotype', 'environment', 'Trial_order', SLOW_COL]].rename(columns={SLOW_COL: 'PAC'})
slow['gamma_band'] = 'slow'
fast = df[['mouse_name', 'Genotype', 'environment', 'Trial_order', FAST_COL]].rename(columns={FAST_COL: 'PAC'})
fast['gamma_band'] = 'fast'

long_pac = pd.concat([slow, fast], ignore_index=True).dropna(subset=['PAC', 'Trial_order'])
long_pac['mouse_name']  = long_pac['mouse_name'].astype(str)
long_pac['Genotype']    = long_pac['Genotype'].astype(str)
long_pac['environment'] = long_pac['environment'].astype(str)
long_pac['gamma_band']  = long_pac['gamma_band'].astype(str)
long_pac['Trial_order'] = long_pac['Trial_order'].astype(str)

print(f"n rows = {len(long_pac)}, n mice = {long_pac['mouse_name'].nunique()}")
print(long_pac.groupby(['environment', 'Trial_order', 'Genotype', 'gamma_band'])['PAC'].agg(['count', 'median']))

# ── Pass to R and fit model ───────────────────────────────────────────────────
with localconverter(ro.default_converter + pandas2ri.converter):
    r_df = ro.conversion.py2rpy(long_pac)
ro.globalenv['dat'] = r_df

r_code = """
suppressWarnings({
    dat$mouse_name  <- factor(dat$mouse_name)
    dat$Genotype    <- factor(dat$Genotype,    levels = c("WT", "NLGF"))
    dat$gamma_band  <- factor(dat$gamma_band,  levels = c("slow", "fast"))
    dat$environment <- factor(dat$environment)
    dat$Trial_order <- factor(dat$Trial_order, levels = c("first", "second"))

    contrasts(dat$Genotype)    <- contr.sum(nlevels(dat$Genotype))
    contrasts(dat$gamma_band)  <- contr.sum(nlevels(dat$gamma_band))
    contrasts(dat$environment) <- contr.sum(nlevels(dat$environment))
    contrasts(dat$Trial_order) <- contr.sum(nlevels(dat$Trial_order))

    fit <- lmer(
        PAC ~ Genotype * gamma_band * environment * Trial_order + (1 | mouse_name),
        data = dat, REML = FALSE
    )

    aov_table <- as.data.frame(anova(fit, type = 3))
    aov_table <- data.frame(term = rownames(aov_table), aov_table,
                            row.names = NULL, check.names = FALSE)

    # 1. Genotype (NLGF vs WT) within each environment × Trial_order
    emm_geno <- emmeans(fit, ~ Genotype | environment * Trial_order)
    geno_df  <- as.data.frame(contrast(emm_geno, method = "pairwise"))

    # 2. Trial_order effect within each Genotype × environment
    emm_trial <- emmeans(fit, ~ Trial_order | Genotype * environment)
    trial_df  <- as.data.frame(contrast(emm_trial, method = "pairwise"))

    # 3. Environment effect within each Genotype × Trial_order
    emm_env <- emmeans(fit, ~ environment | Genotype * Trial_order)
    env_df  <- as.data.frame(contrast(emm_env, method = "pairwise"))

    re_var  <- as.numeric(VarCorr(fit)$mouse_name[1,1])
    aic_val <- AIC(fit)

    list(
        anova_table     = aov_table,
        geno_contrasts  = geno_df,
        trial_contrasts = trial_df,
        env_contrasts   = env_df,
        re_var          = re_var,
        aic             = aic_val
    )
})
"""
out = ro.r(r_code)

with localconverter(ro.default_converter + pandas2ri.converter):
    aov_df          = ro.conversion.rpy2py(out.rx2('anova_table'))
    geno_contrasts  = ro.conversion.rpy2py(out.rx2('geno_contrasts'))
    trial_contrasts = ro.conversion.rpy2py(out.rx2('trial_contrasts'))
    env_contrasts   = ro.conversion.rpy2py(out.rx2('env_contrasts'))

re_var  = float(out.rx2('re_var')[0])
aic_val = float(out.rx2('aic')[0])

print(f"\nAIC: {aic_val:.2f}  |  RE variance (mouse): {re_var:.6f}")

print("\n─── Type III ANOVA (lmerTest, contr.sum) ─────────────────────────────────────")
print(aov_df.to_string(index=False))

print("\n─── 1. Genotype (NLGF vs WT) within each environment × Trial_order ──────────")
print(geno_contrasts.to_string(index=False))

print("\n─── 2. Trial_order (first vs second) within each Genotype × environment ──────")
print(trial_contrasts.to_string(index=False))

print("\n─── 3. Environment within each Genotype × Trial_order ────────────────────────")
print(env_contrasts.to_string(index=False))

# ── Save ──────────────────────────────────────────────────────────────────────
out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

aov_df.to_csv(out_dir / "PAC_lmer_4way_anova_table.csv", index=False)
geno_contrasts.to_csv(out_dir / "PAC_lmer_4way_geno_contrasts.csv", index=False)
trial_contrasts.to_csv(out_dir / "PAC_lmer_4way_trial_contrasts.csv", index=False)
env_contrasts.to_csv(out_dir / "PAC_lmer_4way_env_contrasts.csv", index=False)

print(f"\n✅ Saved to: {OUTPUT_DIR}")

## MLM metric ~ Genotype * environment + (1 | mouse_name)


In [ ]:
"""
LFP analysis:
- Mixed linear models:
    metric ~ Genotype * environment + (1 | mouse_name)

- Violin plots:
    raw data shown as violins/points
    significance stars come from mixed-model genotype contrasts
    within each environment
"""

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

from scipy.stats import norm
import statsmodels.formula.api as smf


# ─── Journal / Illustrator global styles ──────────────────────────────────────
mpl.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "Arial",
    "axes.grid": False,
    "font.size": 7,
    "axes.labelsize": 7,
    "axes.titlesize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "legend.fontsize": 6,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "ytick.left": True,
    "ytick.direction": "in",
    "savefig.transparent": True,
    "savefig.bbox": "tight",
})

GENO_ORDER  = ["WT", "NLGF"]
GENO_COLORS = {"NLGF": "#E07B54", "WT": "#5B8DB8"}


def _pstars(p):
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


# ─── Load data ────────────────────────────────────────────────────────────────
basepath = Path(
    "/Users/loukia/Library/CloudStorage/Dropbox-UCL/"
    "Loukia Katsouri/DataProtocolsEquipment/Ephys_Analysis/"
    "RobinData/Analysis/LFP_analysis"
)

LFP_CSV_PATH = basepath / "concatenated_lfp_stats.csv"
OUTPUT_DIR = LFP_CSV_PATH.parent / "LFP_analysis_mixed_model_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(LFP_CSV_PATH)

df["Genotype"] = pd.Categorical(
    df["Genotype"],
    categories=GENO_ORDER,
    ordered=True
)

df["environment"] = df["environment"].astype(str)
df["mouse_name"] = df["mouse_name"].astype(str)

if "Experimenter" in df.columns:
    df["Experimenter"] = df["Experimenter"].astype(str)


# ─── Metrics to analyse ───────────────────────────────────────────────────────
metrics = {
    "max_theta_freq": "Theta peak frequency (Hz)",
    "comodulogram_PAC_slow_gamma": "Theta–slow gamma PAC",
    "comodulogram_PAC_fast_gamma": "Theta–fast gamma PAC",
    "n_oscillatory_epochs_slow_gamm": "Slow gamma event count",
    "n_oscillatory_epochs_fast_gamma": "Fast gamma event count",
    "theta_running_slope": "Theta–running slope",
    "theta_running_intercept": "Theta–running intercept",
    "theta_running_rvalue": "Theta–running r",
}

metrics = {k: v for k, v in metrics.items() if k in df.columns}


# ─── Mixed models + LMM genotype contrasts for plotting ───────────────────────
mixed_results = []
lmm_plot_pvals = []

for metric in metrics:

    model_df = df[[
        metric,
        "Genotype",
        "environment",
        "mouse_name"
    ]].dropna()

    if model_df["Genotype"].nunique() < 2:
        continue

    if model_df["environment"].nunique() < 2:
        continue

    try:
        model = smf.mixedlm(
            f"{metric} ~ Genotype * environment",
            data=model_df,
            groups=model_df["mouse_name"]
        )

        fit = model.fit(reml=False, method="lbfgs")

        # Save full model terms
        for term, beta in fit.params.items():
            mixed_results.append({
                "metric": metric,
                "term": term,
                "beta": beta,
                "se": fit.bse.get(term, np.nan),
                "z": fit.tvalues.get(term, np.nan),
                "pvalue": fit.pvalues.get(term, np.nan),
                "stars": _pstars(fit.pvalues.get(term, np.nan)),
                "n": model_df.shape[0],
                "n_mice": model_df["mouse_name"].nunique(),
            })

        # Environment-specific NLGF vs WT contrasts from the LMM
        params = fit.params
        cov = fit.cov_params()

        for env in sorted(model_df["environment"].dropna().unique()):

            contrast = pd.Series(0.0, index=params.index)

            if "Genotype[T.NLGF]" in contrast.index:
                contrast["Genotype[T.NLGF]"] = 1.0

            interaction_term = f"Genotype[T.NLGF]:environment[T.{env}]"

            if interaction_term in contrast.index:
                contrast[interaction_term] = 1.0

            beta = float(np.dot(contrast, params))
            se = float(np.sqrt(np.dot(contrast, np.dot(cov, contrast))))

            if se > 0:
                z = beta / se
                p = 2 * (1 - norm.cdf(abs(z)))
            else:
                z = np.nan
                p = np.nan

            lmm_plot_pvals.append({
                "metric": metric,
                "environment": env,
                "contrast": "NLGF_vs_WT",
                "beta": beta,
                "se": se,
                "z": z,
                "pvalue": p,
                "stars": _pstars(p),
                "n": model_df.loc[
                    model_df["environment"] == env
                ].shape[0],
                "n_mice": model_df.loc[
                    model_df["environment"] == env,
                    "mouse_name"
                ].nunique(),
            })

        clean_metric = metric.replace("/", "_")

        with open(
            os.path.join(OUTPUT_DIR, f"mixed_model_{clean_metric}.txt"),
            "w"
        ) as f:
            f.write(str(fit.summary()))

    except Exception as e:
        mixed_results.append({
            "metric": metric,
            "term": "MODEL_FAILED",
            "beta": np.nan,
            "se": np.nan,
            "z": np.nan,
            "pvalue": np.nan,
            "stars": "",
            "n": model_df.shape[0],
            "n_mice": model_df["mouse_name"].nunique(),
            "error": str(e),
        })


mixed_results = pd.DataFrame(mixed_results)
mixed_results.to_csv(
    os.path.join(OUTPUT_DIR, "mixed_model_results.csv"),
    index=False
)

lmm_plot_pvals = pd.DataFrame(lmm_plot_pvals)
lmm_plot_pvals.to_csv(
    os.path.join(OUTPUT_DIR, "mixed_model_plot_pvalues.csv"),
    index=False
)

# ─── Plotting helper: raw data violins + MLM genotype contrasts ──────────────
def plot_metric_violin_mlm(data, metric, ylabel, save=True):
    env_order = ["open_field", "linear_track"]
    envs = [env for env in env_order if env in data["environment"].dropna().unique()]

    if len(envs) == 0:
        return
    fig, axes = plt.subplots(
        1,
        len(envs),
        figsize=(3.3, 1.65),
        sharey=True,
    )

    if len(envs) == 1:
        axes = [axes]

    rng = np.random.default_rng(10)

    for ax, env in zip(axes, envs):
        sub = data.loc[data["environment"] == env].copy()
        sub = sub[["Genotype", "mouse_name", metric]].dropna()

        genos = [
            g for g in GENO_ORDER
            if g in sub["Genotype"].astype(str).values
        ]

        if len(genos) < 2:
            ax.set_visible(False)
            continue

        vals = {
            g: sub.loc[sub["Genotype"].astype(str) == g, metric].values
            for g in genos
        }

        # if all(len(vals[g]) >= 3 for g in genos):
        #     with warnings.catch_warnings():
        #         warnings.simplefilter("ignore")
        #         parts = ax.violinplot(
        #             [vals[g] for g in genos],
        #             positions=list(range(len(genos))),
        #             showmedians=False,
        #             showextrema=False,
        #             widths=0.8,
        #         )

        #     for i_v, pc in enumerate(parts["bodies"]):
        #         pc.set_facecolor(GENO_COLORS.get(genos[i_v], "#aaaaaa"))
        #         pc.set_alpha(0.45)
        #         pc.set_edgecolor("none")

        for xi, g in enumerate(genos):
            if len(vals[g]) == 0:
                continue

            q1 = np.nanpercentile(vals[g], 25)
            med = np.nanmedian(vals[g])
            q3 = np.nanpercentile(vals[g], 75)

            ax.hlines(
                [q1, q3],
                xi - 0.16,
                xi + 0.16,
                colors="black",
                linewidth=0.8,
                zorder=7,
            )
            ax.hlines(
                med,
                xi - 0.24,
                xi + 0.24,
                colors="black",
                linewidth=1.6,
                zorder=8,
            )

            jitter = rng.normal(0, 0.048, size=len(vals[g]))
            ax.scatter(
                np.full(len(vals[g]), xi) + jitter,
                vals[g],
                s=10,
                color=GENO_COLORS.get(g, "#aaaaaa"),
                edgecolor="black",
                linewidth=0.15,
                alpha=0.9,
                zorder=10,
            )

        # MLM annotation from the mixed-model cell above
        p_row = lmm_plot_pvals[
            (lmm_plot_pvals["metric"] == metric) &
            (lmm_plot_pvals["environment"] == env) &
            (lmm_plot_pvals["contrast"] == "NLGF_vs_WT")
        ]

        if len(p_row):
            p = p_row["pvalue"].iloc[0]
            stars = _pstars(p)

            ymax = np.nanmax(sub[metric].values)
            ymin = np.nanmin(sub[metric].values)
            yrng = ymax - ymin if ymax > ymin else max(abs(ymax) * 0.1, 1.0)

            y = ymax + 0.12 * yrng
            h = 0.04 * yrng

            ax.plot(
                [0, 0, 1, 1],
                [y, y + h, y + h, y],
                color="black",
                linewidth=0.8,
                clip_on=False,
            )
            ax.text(
                0.5,
                y,
                stars,
                ha="center",
                va="bottom",
                fontsize=12,
                fontweight="bold",
            )

        ax.set_xticks(range(len(genos)))
        ax.set_xticklabels(genos)
        ax.set_title(env)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    axes[0].set_ylabel(ylabel)
    fig.suptitle(ylabel, y=0.92, fontsize=7, fontweight="bold")
    fig.tight_layout()

    if save:
        clean_metric = metric.replace("/", "_")
        fig.savefig(os.path.join(OUTPUT_DIR, f"{clean_metric}_violin_LMM.pdf"))

    plt.show()
    plt.close(fig)


# ─── Make all violin plots ────────────────────────────────────────────────────
for metric, ylabel in metrics.items():
    plot_metric_violin_mlm(df, metric, ylabel)

print("Done.")
print(f"Saved results and figures to: {OUTPUT_DIR}")

## Compare MLM vs Mann-Whitney U: concordance of significance 



Several reasons why the MLM is better suited to your data:

**1. Handles repeated measures properly**
Each mouse contributes data from both `open_field` and `linear_track`. Mann-Whitney treats all observations as independent — it doesn't know that two rows belong to the same mouse. The `(1|mouse_name)` random intercept explicitly models the correlation between repeated measurements from the same animal, giving you correct standard errors.

**2. Tests environment and genotype simultaneously**
Mann-Whitney only compares WT vs NLGF within one environment at a time. The MLM fits both environments in one model, so the `Genotype * environment` interaction term tells you directly whether **the genotype difference changes between environments** — something Mann-Whitney cannot answer at all.

**3. Borrows strength across environments**
By pooling all data into one model, the MLM uses more data when estimating genotype effects, giving better-powered estimates, especially when per-environment sample sizes are small.

**4. Accounts for mouse-level variability**
If some mice just have systematically higher/lower PAC regardless of genotype (which is very likely in neural data), the random intercept absorbs that variability. Mann-Whitney lumps it into the error, inflating noise.

**5. Effect size estimate**
The MLM gives you a `beta` (the estimated magnitude of the genotype difference in original units), not just a p-value. This is more informative for biological interpretation.

---

**When Mann-Whitney is still useful here:**
- Your data are non-normal (Shapiro-Wilk showed this) — Mann-Whitney makes no distributional assumptions, whereas the MLM assumes normally distributed residuals. With small n, this matters.
- It's simpler to explain and widely accepted in neuroscience.
- The concordance cell below compares both — if they agree, you have convergent evidence.

**Bottom line:** For your design (same mice, two environments, two genotypes), the MLM is statistically more appropriate. Mann-Whitney is a useful sanity check, but it ignores the repeated-measures structure.

In [ ]:
# ── Compare MLM vs Mann-Whitney U: concordance of significance ───────────────
from IPython.display import display, HTML

ALPHA = 0.05

# Mixed linear model: environment-specific NLGF vs WT contrasts
mlm = lmm_plot_pvals[
    lmm_plot_pvals["contrast"] == "NLGF_vs_WT"
][[
    "metric",
    "environment",
    "beta",
    "se",
    "z",
    "pvalue",
    "n",
    "n_mice",
]].copy()

mlm = mlm.rename(columns={
    "beta": "mlm_beta",
    "se": "mlm_se",
    "z": "mlm_z",
    "pvalue": "mlm_p",
    "n": "mlm_n",
    "n_mice": "mlm_n_mice",
})
mlm["mlm_sig"] = mlm["mlm_p"] < ALPHA

# Mann-Whitney
mwu = mw_results[[
    "metric",
    "environment",
    "n_WT",
    "n_NLGF",
    "WT_median",
    "NLGF_median",
    "mannwhitney_U",
    "pvalue",
]].copy()

mwu = mwu.rename(columns={
    "pvalue": "mwu_p",
})
mwu["mwu_sig"] = mwu["mwu_p"] < ALPHA

# Merge on metric × environment
comp = pd.merge(
    mlm,
    mwu,
    on=["metric", "environment"],
    how="outer",
    indicator=True,
)

# Classification
def classify(row):
    g = bool(row.get("mlm_sig")) if pd.notna(row.get("mlm_sig")) else False
    m = bool(row.get("mwu_sig")) if pd.notna(row.get("mwu_sig")) else False

    if g and m:
        return "both"
    if g:
        return "MLM only"
    if m:
        return "MWU only"
    return "neither"

comp["concordance"] = comp.apply(classify, axis=1)

# p-value formatter
def _fmt_p(p):
    if pd.isna(p):
        return "—"
    if p < 0.001:
        return f"{p:.4f} ***"
    if p < 0.01:
        return f"{p:.4f} **"
    if p < 0.05:
        return f"{p:.4f} *"
    return f"{p:.4f}"

# Force environment display order
env_order = ["open_field", "linear_track"]
comp["environment"] = pd.Categorical(
    comp["environment"],
    categories=env_order,
    ordered=True
)

# ── Display: summary by concordance ───────────────────────────────────────────
display(HTML("<h3>Mixed Linear Model vs Mann-Whitney U — concordance of significance</h3>"))

for label in ["both", "MLM only", "MWU only", "neither"]:
    grp = comp[comp["concordance"] == label].copy()
    if grp.empty:
        continue

    display(HTML(f"<h4>{label} ({len(grp)} comparisons)</h4>"))

    show = grp[[
        "metric",
        "environment",
        "mlm_p",
        "mlm_beta",
        "mlm_se",
        "mlm_z",
        "mwu_p",
        "mannwhitney_U",
        "WT_median",
        "NLGF_median",
        "n_WT",
        "n_NLGF",
    ]].sort_values(["environment", "metric"])

    def _colour_row(row):
        g = row.get("mlm_p", np.nan)
        m = row.get("mwu_p", np.nan)

        g_sig = pd.notna(g) and g < ALPHA
        m_sig = pd.notna(m) and m < ALPHA

        if g_sig and m_sig:
            bg = "#c8e6c9"
        elif g_sig or m_sig:
            bg = "#fff9c4"
        else:
            bg = ""

        return [f"background-color:{bg}" if bg else "" for _ in row]

    display(
        show.style
            .apply(_colour_row, axis=1)
            .format({
                "mlm_p": "{:.4f}",
                "mlm_beta": "{:.4f}",
                "mlm_se": "{:.4f}",
                "mlm_z": "{:.3f}",
                "mwu_p": "{:.4f}",
                "mannwhitney_U": "{:.2f}",
                "WT_median": "{:.4f}",
                "NLGF_median": "{:.4f}",
            })
            .set_properties(**{"font-size": "11px"})
    )

# ── Quick count summary ───────────────────────────────────────────────────────
print("\nConcordance counts")
print(comp["concordance"].value_counts().to_string())

# Optional compact summary by environment
print("\nConcordance by environment")
print(pd.crosstab(comp["environment"], comp["concordance"]).to_string())

# ── Save ──────────────────────────────────────────────────────────────────────
comp_path = os.path.join(OUTPUT_DIR, "mlm_vs_mwu_concordance.csv")
comp[[
    "metric",
    "environment",
    "mlm_p",
    "mlm_beta",
    "mlm_se",
    "mlm_z",
    "mlm_n",
    "mlm_n_mice",
    "mwu_p",
    "mannwhitney_U",
    "n_WT",
    "n_NLGF",
    "WT_median",
    "NLGF_median",
    "concordance",
]].to_csv(comp_path, index=False)

print(f"\nSaved: {comp_path}")